# Fase 2 · Pipeline de obtención, limpieza, transformación y validación

**MCDI500 · Programación para la Ciencia de Datos** · Grupo 6

Cada decisión se toma **después de medir** y se registra en la bitácora con sus cifras.
Regla de oro: no aplicar una técnica porque es la habitual, sino la que las cifras justifican.

In [ ]:
from pathlib import Path
import os, sys
RAIZ = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ); sys.path.insert(0, str(RAIZ))
import numpy as np, pandas as pd
from src.bitacora import registrar, exportar

df = pd.read_csv("data/processed/titulados_2025_pregrado_univ.csv", sep=";", low_memory=False)
print(df.shape)

## Paso 6 · Exploración: medir antes de tocar nada

In [ ]:
# 1) dimensiones y tipos
print(df.dtypes.value_counts())
# 2) valores faltantes y códigos de relleno
nulos = df.isna().mean().mul(100).round(2).sort_values(ascending=False)
print(nulos[nulos > 0])
print("anio_ing_carr_ori == 1900:", (df.anio_ing_carr_ori == 1900).sum())
print("sem_ing_carr_ori == 0    :", (df.sem_ing_carr_ori == 0).sum())
# 3) categorías observadas (misma categoría escrita de dos maneras)
for c in ["jornada", "modalidad", "tipo_inst_2", "rango_edad", "nivel_carrera_1"]:
    print(c, "→", sorted(df[c].dropna().unique()))
# 4) valores atípicos con rango intercuartílico
for c in ["dur_estudio_carr", "dur_proceso_tit", "dur_total_carr"]:
    q1, q3 = df[c].quantile([0.25, 0.75]); ric = q3 - q1
    atip = ((df[c] < q1 - 1.5 * ric) | (df[c] > q3 + 1.5 * ric)).sum()
    print(f"{c}: Q1={q1} Q3={q3} RIC={ric} atípicos={atip:,} ({100*atip/len(df):.1f} %)")

## Paso 7 · Limpieza e imputación (comparar antes de decidir)

- Duplicados: eliminar (3 filas).
- `anio_ing_carr_ori = 1900` y `sem_ing_carr_ori = 0`: tratar como faltante; comparar eliminación vs imputación por mediana midiendo el cambio en la desviación estándar.
- `nombre_grado`: 8 % nulos → decidir si se conserva como texto o se descarta.
- Registrar cada decisión con `registrar("F2 · limpieza", "...")`.

In [ ]:
# TODO: implementar en funciones dentro de src/ (con docstrings y manejo de excepciones)
# registrar("F2 · limpieza", f"duplicados: {df.duplicated().sum()} filas eliminadas")

## Paso 8 · Transformación por tipo de variable

- Nominales (`tipo_inst_2`, `jornada`, `modalidad`, `area_conocimiento`, `region_sede`): one-hot encoding.
- Ordinales: declarar el orden explícitamente, p. ej. `rango_edad`: 15 a 19 < 20 a 24 < 25 a 29 < 30 a 34 < 35 a 39 < 40 y más.
- Fecha `fecha_obtencion_titulo` (AAAAMMDD): parsear y derivar mes; `fec_nac_alu` (AAAAMM): derivar edad al titularse.
- Alta cardinalidad (`nomb_carrera`, `nomb_inst`): agrupar categorías poco frecuentes.
- Identificador `mrun`: verificar unicidad y excluir del análisis.

In [ ]:
# TODO

## Paso 9 · Escalamiento (comparar StandardScaler, MinMaxScaler y RobustScaler; elegir con coherencia respecto de la imputación)

In [ ]:
# TODO

## Paso 10 · Validación: demostrar, no afirmar

In [ ]:
# assert df_final.isna().sum().sum() == 0
# assert not df_final.duplicated().any()
# pruebas de las funciones de src/: caso normal, caso límite, excepción

## Paso 11 · Persistencia y bitácora

In [ ]:
# df_final.to_csv("data/processed/titulados_2025_pregrado_univ_limpio.csv", sep=";", index=False)
# verificación: releer y comprobar forma y columnas
# exportar("docs/bitacora.md")